## Análisis y limpieza de `curses.csv` (carreirasgalegas)

Cargamos la tabla de carreras/modalidades (una fila por cada modalidad de cada competición) y le hacemos una primera inspección antes de limpiarla.

In [1]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("../../data/raw/carreirasgalegas/DF_CARREIRASGALEGAS_SUCIO.csv")
curses = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

print("Files x columnes:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Files x columnes: (3192, 14)

competition_id            object
data                      object
distancia_m                int64
is_closed                   bool
lloc                      object
modalitat_nom             object
nom_cursa                 object
ok_d                     float64
ok_h                     float64
ok_total                 float64
race_id                   object
te_pdf_resultats            bool
te_resultats_digitals       bool
total_resultats            int64
dtype: object


,competition_id,data,distancia_m,is_closed,lloc,modalitat_nom,nom_cursa,ok_d,ok_h,ok_total,race_id,te_pdf_resultats,te_resultats_digitals,total_resultats
0,6lZJWbl1,2026-08-02,7000,False,Cabanas,ABSOLUTA,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,40.0,150.0,190.0,4jorQ79Y,True,True,190
1,6lZJWbl1,2026-08-02,3600,False,Cabanas,SUB 18 E SUB 16,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,2.0,8.0,10.0,Kj6GqNlR,True,True,10
2,6lZJWbl1,2026-08-02,1800,False,Cabanas,SUB 14,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,9.0,5.0,14.0,vlvw6kmy,True,True,14
3,6lZJWbl1,2026-08-02,1000,False,Cabanas,SUB 12 E SUB 10,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,21.0,23.0,44.0,yjPVvyme,True,True,44
4,6lZJWbl1,2026-08-02,300,False,Cabanas,PEQUES,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,NaN,NaN,NaN,pld8LDlR,True,False,0


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados y rangos raros
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print("Filas duplicadas por (competition_id, race_id):",
      curses.duplicated(subset=["competition_id", "race_id"]).sum())
print()

print("Rango de fechas (texto):", curses["data"].min(), "->", curses["data"].max())
print()

print("distancia_m <= 0 o nula:", (curses["distancia_m"].fillna(0) <= 0).sum())
print(curses["distancia_m"].describe())
print()

print("total_resultats vs suma de recuentos por estado (ok/dnf/dns...) debería coincidir:")
estat_cols = [c for c in curses.columns if c.endswith("_total")]
print("columnas de estado encontradas:", estat_cols)

Valores nulos por columna:
competition_id              0
data                        0
distancia_m                 0
is_closed                   0
lloc                        0
modalitat_nom               0
nom_cursa                   0
ok_d                     2184
ok_h                     2177
ok_total                 2038
race_id                     0
te_pdf_resultats            0
te_resultats_digitals       0
total_resultats             0
dtype: int64

Filas completamente duplicadas: 0
Filas duplicadas por (competition_id, race_id): 0

Rango de fechas (texto): 2019-04-27 -> 2026-08-02

distancia_m <= 0 o nula: 59
count      3192.000000
mean       4049.389098
std        7076.363073
min           0.000000
25%         637.500000
50%        1609.000000
75%        5000.000000
max      109000.000000
Name: distancia_m, dtype: float64

total_resultats vs suma de recuentos por estado (ok/dnf/dns...) debería coincidir:
columnas de estado encontradas: ['ok_total']


In [3]:
# Quitamos duplicados exactos ANTES de seleccionar columnas — con "race_id"
# todavía presente (el identificador único real de cada subcursa). Hacerlo
# después, como se hacía antes, es un error: dos subcarreras con "race_id"
# distinto pero que coinciden por casualidad en nombre/fecha/modalidad/
# distancia/finishers (p.ej. dos categorías "DISCAPACIDADE" separadas del
# mismo evento) se confundirían con duplicados y se perdería una de las
# dos, aunque sean filas legítimas y distintas. El diagnóstico de arriba
# ya lo confirma: 0 duplicados completos y 0 duplicados por
# (competition_id, race_id) — así que este paso no debería quitar nada
# ahora mismo, pero se deja como red de seguridad para futuras cargas.
antes = len(curses)
curses = curses.drop_duplicates().reset_index(drop=True)
print(f"{antes - len(curses)} filas duplicadas eliminadas ({antes} -> {len(curses)})")

0 filas duplicadas eliminadas (3192 -> 3192)


In [4]:
# Limpieza: nos quedamos solo con las columnas que interesan, renombradas,
# distancia en km (con decimales, aquí hay carreras de solo 300m) y sin
# decimales en finisher_d/finisher_h (salen como float por los NaN cuando
# una carrera no tiene ningún acabado "ok" de ese género). "lloc" (el
# único dato de ubicación que trae esta fuente) se recupera como
# "municipio" — antes se descartaba sin usar.
curses_limpio = curses[
    ["competition_id", "nom_cursa", "data", "distancia_m", "modalitat_nom", "ok_d", "ok_h", "lloc"]
].rename(columns={
    "competition_id": "id",
    "nom_cursa": "nombre_carrera",
    "data": "fecha",
    "modalitat_nom": "modalidad",
    "ok_d": "finisher_d",
    "ok_h": "finisher_h",
    "lloc": "municipio",
})

curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"])
curses_limpio["distancia"] = curses_limpio["distancia_m"] / 1000
curses_limpio = curses_limpio.drop(columns=["distancia_m"])

curses_limpio[["finisher_d", "finisher_h"]] = curses_limpio[["finisher_d", "finisher_h"]].fillna(0).astype(int)

print(curses_limpio.shape)
curses_limpio.head()

(3192, 8)


,id,nombre_carrera,fecha,modalidad,finisher_d,finisher_h,municipio,distancia
0,6lZJWbl1,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,2026-08-02,ABSOLUTA,40,150,Cabanas,7.0
1,6lZJWbl1,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,2026-08-02,SUB 18 E SUB 16,2,8,Cabanas,3.6
2,6lZJWbl1,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,2026-08-02,SUB 14,9,5,Cabanas,1.8
3,6lZJWbl1,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,2026-08-02,SUB 12 E SUB 10,21,23,Cabanas,1.0
4,6lZJWbl1,XXXIII CROS POPULAR PIÑEIRAL DE CABANAS 2026,2026-08-02,PEQUES,0,0,Cabanas,0.3


### Cómo se clasifica `modalidad`

El campo original `modalidad` mezcla dos cosas distintas según la carrera: a veces indica la **disciplina** (running de asfalto, trail, ciclismo/BTT, marcha...) y a veces indica el **público/edad** (SUB 14, Infantil, Máster...). Para poder analizar cada cosa por separado, se divide en dos columnas nuevas:

1. **`tipo_modalidad`** (disciplina): se busca en el texto de `modalidad` un conjunto de palabras clave por categoría (p.ej. "trail", "btt", "cross", "milla"...). Si `modalidad` no da ninguna pista (lo habitual aquí, ya que suele contener solo la categoría de edad), se prueba con el **nombre de la carrera** (`nombre_carrera`), porque muchos eventos llevan la disciplina en su propio nombre ("CARREIRA...", "CROS...", "TRAIL..."). Si ni una cosa ni la otra dan ninguna pista y `modalidad` estaba vacía, se asume **road running** por defecto (el formato más común en este sitio); si `modalidad` sí tenía texto pero no reconocible, queda en **"Otros"**.

2. **`publico`** (edad/audiencia): se buscan palabras clave de edad, también con respaldo en `nombre_carrera`. Las categorías Infantil, Cadete y Juvenil se fusionan en una única categoría, **"Infantil"** (todo lo que no es adulto abierto ni veterano). Antes de asumir nada, se descartan explícitamente los términos que NO son edad (categorías de discapacidad, artefactos administrativos como "inscrición"/"participación", basura de scraping, o etiquetas de distancia+género sin pista de edad como "11 K"), que van a **"Otros"**. Si no hay ninguna marca de edad reconocible, se asume **Absoluta/General** por defecto (la mayoría de los casos sin calificativo son así).

3. Al final, se **fusionan las filas** que comparten evento (`nombre_carrera`+`fecha`), `distancia`, `tipo_modalidad` y `publico` — normalmente son la misma categoría repartida en sub-categorías (p.ej. SUB8/SUB10/SUB12... todas "Infantil"), así que se colapsan en una sola fila sumando `ok_d`/`ok_h`.

Este es un proceso iterativo por palabras clave: no es infalible, así que cada paso imprime lo que ha quedado sin clasificar para poder revisarlo y afinar las reglas.

In [5]:
# Clasificamos modalidad en 5 categorías por palabras clave. Si "modalidad"
# no da ninguna pista (aquí suele ser solo la categoría de edad, p.ej.
# "SUB 14"), probamos con el nombre de la carrera — muchos eventos llevan
# la disciplina en el propio nombre ("CARREIRA...", "CROS...", "TRAIL...")
# aunque la modalidad en sí no diga nada.
import re

_CATEGORIAS = {
    "trail running": r"trail|trekking|vertical|\btra\b|\bkv\b|\butpd\b",
    "Ciclismo y btt": r"btt|ciclis|bici|mtb|gravel|ciclotur|e-?bike|\bbike\b|gran ?fondo",
    "Multidisciplina": r"duatl|triatl|multidep|multidisci|aquatl|acuatl|combinada",
    "road running": r"carrera|carreira|running|popular|asfalto|ruta|marat|cross|cros|absoluta|10k|5k|21k|half|corredor|corre|lasterketa|campo a trav[eé]s|milla",
    "marcha": r"martxa|marcha|\bmar\b|andarin|andaina",
}

def _clasificar_texto(texto):
    for categoria, patron in _CATEGORIAS.items():
        if re.search(patron, texto):
            return categoria
    return None

def _clasificar(row):
    modalidad = row["modalidad"]
    texto_modalidad = "" if pd.isna(modalidad) else modalidad.lower().strip()
    categoria = _clasificar_texto(texto_modalidad) if texto_modalidad else None
    if categoria:
        return categoria

    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre.lower()
    categoria = _clasificar_texto(texto_nombre)
    if categoria:
        return categoria

    if texto_modalidad == "":
        # Ni la modalidad (vacía) ni el nombre de la carrera dan ninguna
        # pista — por defecto asumimos road running.
        return "road running"
    return "Otros"

curses_limpio["tipo_modalidad"] = curses_limpio.apply(_clasificar, axis=1)

print(curses_limpio["tipo_modalidad"].value_counts())
print()
print("Valores de 'modalidad' que han caído en 'Otros' (revisar si falta alguna palabra clave):")
print(curses_limpio.loc[curses_limpio["tipo_modalidad"] == "Otros", "modalidad"].value_counts())
print()
print("Nombres de carrera de las filas que siguen en 'Otros' (para ver qué patrón añadir):")
print(curses_limpio.loc[curses_limpio["tipo_modalidad"] == "Otros", "nombre_carrera"].value_counts().head(30))

tipo_modalidad
road running       2486
Otros               317
marcha              233
trail running       152
Ciclismo y btt        3
Multidisciplina       1
Name: count, dtype: int64

Valores de 'modalidad' que han caído en 'Otros' (revisar si falta alguna palabra clave):
modalidad
SUB 10             28
SUB 8              27
SUB 12             22
SUB 14             21
PITUFOS            18
                   ..
PITUFO              1
SUB 14 - SUB 16     1
SUB 10 - SUB 12     1
MÁSTER              1
SENIOR              1
Name: count, Length: 103, dtype: int64

Nombres de carrera de las filas que siguen en 'Otros' (para ver qué patrón añadir):
nombre_carrera
LVI  TROFEO “JOAQUÍN ROMERO” VI MEMORIAL                   10
XIII NIGRÁN AREA 2024                                      10
XXXV MEMORIAL ADOLFO ROS - VOLTA A RÍA                      9
+10 MARÍN MANUEL ROSALES, GRAN PREMIO CONCELLO DE MARÍN     9
XOGADE - FINAL COMARCAL AS BURGAS 2024/2025                 8
XOGADE - FASE PREVIA AS 

In [6]:
# Ya no vaciamos "modalidad": tipo_modalidad y publico usan cada uno sus
# propias palabras clave (no se pisan entre sí), así que dejamos el texto
# original intacto para poder comparar a ojo.
curses_limpio["modalidad"].isna().sum()

np.int64(0)

In [7]:
# Igual que con tipo_modalidad: clasificamos el público al que va dirigida
# la carrera por palabras clave, y si "modalidad" no da ninguna pista útil
# (vacía, o es una etiqueta administrativa/basura de scraping), probamos
# también con el nombre de la carrera antes de rendirnos.
_OTROS_PATRON = (
    r"discapacidad|invidente|handbike|cadeira de rodas|"
    r"inscrici[oó]n|participaci[oó]n|promoci[oó]n|simulaci[oó]n|"
    r"material selecci[oó]n|^pago$|prueba de camiseta|asdfsadf|dsfasdfasd|"
    r"^\d+\s?k(m)?\b|^\d+\s?(fem|masc)\b"
)
_EQUIPOS_PATRON = r"equipos?\b|equips?\b"

_PUBLICOS = {
    "Infantil": (
        r"infant|benjam|benxam|alev|prebenjam|prebenxam|chupet|peque|pitufo|"
        r"biber[oó]n|familiar|menores|escolar|años|"
        r"sub\s?\d+\b|"
        r"cadete|juvenil|xuvenil|junior|cadet\b|promesa|preuniversitari"
    ),
    "Mayores/Veteranos": r"mayores|veteran|master|m[aá]ster",
    "Elite": r"\belit|profesional",
    "Absoluta/General": r"absoluta|\babs\b|general|popular|senior|adultos?|\bopen\b",
}

def _clasificar_publico_texto(texto):
    for publico, patron in _PUBLICOS.items():
        if re.search(patron, texto):
            return publico
    return None

def _clasificar_publico(row):
    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre.lower()
    if re.search(_EQUIPOS_PATRON, texto_nombre):
        return "Equipos"

    modalidad = row["modalidad"]
    texto_modalidad = "" if pd.isna(modalidad) else modalidad.lower().strip()
    if re.search(_EQUIPOS_PATRON, texto_modalidad):
        return "Equipos"

    publico = _clasificar_publico_texto(texto_modalidad) if texto_modalidad else None
    if publico:
        return publico

    publico = _clasificar_publico_texto(texto_nombre)
    if publico:
        return publico

    if texto_modalidad and re.search(_OTROS_PATRON, texto_modalidad):
        return "Otros"
    return "Absoluta/General"

curses_limpio["publico"] = curses_limpio.apply(_clasificar_publico, axis=1)

print(curses_limpio["publico"].value_counts())
print()
print("Valores de 'modalidad' que han caído por defecto en 'Absoluta/General' (revisar si alguno debería tener edad propia):")
print(curses_limpio.loc[curses_limpio["publico"] == "Absoluta/General", "modalidad"].value_counts())
print()
print("Valores de 'modalidad' que han caído en 'Otros':")
print(curses_limpio.loc[curses_limpio["publico"] == "Otros", "modalidad"].value_counts())

publico
Infantil             1908
Absoluta/General     1065
Otros                 177
Mayores/Veteranos      40
Elite                   2
Name: count, dtype: int64

Valores de 'modalidad' que han caído por defecto en 'Absoluta/General' (revisar si alguno debería tener edad propia):
modalidad
ANDAINA                              114
ABSOLUTA                              53
TRAIL LONGO                           27
TRAIL CURTO                           27
ADULTOS                               23
                                    ... 
QUENDA 2                               1
QUENDA 1                               1
CIRCUÍTO ADULTOS  + SAN SILVESTRE      1
CARREIRA E ANDAINA ADULTOS             1
MILLA ÉTITE MASC                       1
Name: count, Length: 457, dtype: int64

Valores de 'modalidad' que han caído en 'Otros':
modalidad
10K                          33
5K                           30
DISCAPACIDADE                12
CADEIRA DE RODAS              7
21K                          

In [8]:
# De momento NO borramos "modalidad" (se queda para poder revisar a ojo
# que tipo_modalidad/publico se han asignado bien); se borrará más adelante.
curses_limpio.columns.tolist()

['id',
 'nombre_carrera',
 'fecha',
 'modalidad',
 'finisher_d',
 'finisher_h',
 'municipio',
 'distancia',
 'tipo_modalidad',
 'publico']

In [9]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['id', 'nombre_carrera', 'fecha', 'modalidad', 'finisher_d', 'finisher_h', 'municipio', 'distancia', 'tipo_modalidad', 'publico']
Filas x columnas: (3192, 10)

id                        object
nombre_carrera            object
fecha             datetime64[ns]
modalidad                 object
finisher_d                 int64
finisher_h                 int64
municipio                 object
distancia                float64
tipo_modalidad            object
publico                   object
dtype: object

Cruce tipo_modalidad x publico:
publico          Absoluta/General  Elite  Infantil  Mayores/Veteranos  Otros
tipo_modalidad                                                              
Ciclismo y btt                  2      0         1                  0      0
Multidisciplina                 1      0         0                  0      0
Otros                          16      0       250                  1     50
marcha                        165      0        59                  

,id,nombre_carrera,fecha,modalidad,finisher_d,finisher_h,municipio,distancia,tipo_modalidad,publico
2305,Ov9O1DlL,XIII CARREIRA POPULAR CONCELLO DE O CORGO,2022-10-22,"ABSOLUTA 10,5K",0,0,O Corgo,10.500,road running,Absoluta/General
1981,dxmA2B9Z,VI CARREIRA E ANDAINA SOLIDARIA POLA ESCLEROSE...,2023-06-11,PEQUES,0,0,Santiago de Compostela,0.050,road running,Infantil
2441,Aame5AmZ,CAMPIONATO ABSOLUTO,2022-07-23,INSCRICIÓN A CAMPEONATO,0,0,A Coruña,0.001,Otros,Otros
171,qjyNNXlK,II CARREIRA POPULAR CONCELLO DA BAÑA,2026-05-03,SUB 10,17,5,A Baña,0.800,road running,Infantil
2249,VP92R69o,IX MILLA SOLIDARIA ANEDIA,2022-11-13,MILLA ELITE FEDERADA FEM,0,0,Pontevedra,1.609,road running,Elite
1489,vVjRROj7,"VI CARRERA POPULAR ""PRAIA DE MIÑO"".",2024-05-12,SUB 14,3,6,Miño,2.000,road running,Infantil
2054,V2mJaklM,5ª CARREIRA E ANDAINA CONTRA O CANCRO,2023-04-30,SUB 12,0,0,A Rúa,0.500,road running,Infantil
867,YVjXxNjQ,"III EDICIÓN 10K ""COSTA ÁRTABRA""",2025-03-30,10K,334,676,A Coruña,10.000,road running,Otros
2282,7glY5klV,V MILLA NOCTURNA CONCELLO DE MUGARDOS,2022-10-29,MILLA ABS MASC,0,0,Mugardos,1.609,road running,Absoluta/General
551,yz9zwJjq,II CARREIRA CAMIÑOS DA VÍA VERDE – DEPUTACIÓN ...,2025-09-28,SUB 12,6,12,Ordes - Cerceda,0.000,road running,Infantil


### Carreras con el mismo nombre en la misma fecha

Un mismo evento (mismo `nombre_carrera` + `fecha`) tiene varias filas porque cada una es una modalidad/categoría distinta — normalmente se diferencian en `distancia` y/o en `tipo_modalidad`/`publico`, y eso está bien, es esperable. Si dentro del mismo evento hay filas que coinciden en las tres cosas y solo cambia el `id` (u otra columna), probablemente sean duplicados a fusionar.

In [10]:
# Por cada evento (nombre_carrera + fecha), vemos cuántos valores distintos
# hay de distancia/tipo_modalidad/publico. Si con más de una fila los tres
# son constantes (nunique <= 1), lo único que cambia es el id (u otra
# columna) -> candidatos a fusionar.
resumen_eventos = curses_limpio.groupby(["nombre_carrera", "fecha"]).agg(
    filas=("id", "size"),
    distancias_distintas=("distancia", "nunique"),
    tipos_distintos=("tipo_modalidad", "nunique"),
    publicos_distintos=("publico", "nunique"),
    ids_distintos=("id", "nunique"),
).reset_index()

sospechosos = resumen_eventos[
    (resumen_eventos["filas"] > 1)
    & (resumen_eventos["distancias_distintas"] <= 1)
    & (resumen_eventos["tipos_distintos"] <= 1)
    & (resumen_eventos["publicos_distintos"] <= 1)
].sort_values("filas", ascending=False)

print(f"{len(sospechosos)} eventos con filas que parecen duplicadas "
      f"(misma distancia, tipo_modalidad y publico dentro del mismo evento):")
sospechosos.head(20)

10 eventos con filas que parecen duplicadas (misma distancia, tipo_modalidad y publico dentro del mismo evento):


,nombre_carrera,fecha,filas,distancias_distintas,tipos_distintos,publicos_distintos,ids_distintos
44,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,8,1,1,1,1
513,XOGADE – CROS – FASE PROVINCIAL OURENSE 2025,2025-12-13,8,1,1,1,1
56,CARREIRA E ANDAINA SOLIDARIA GATOCAN OROSO,2026-02-28,3,1,1,1,1
121,I TROFEO JOSÉ LUIS TORRADO FESTA HQR! RÍAS BAIXAS,2023-12-28,3,1,1,1,1
304,ULTRA TRAIL HERÓICA SACRA. CTO. XUNTA DE GALIC...,2025-06-29,3,1,1,1,1
78,I ANDAINA MEMORIAL LUIS NOGUEIRA,2024-08-24,2,1,1,1,1
188,III MEDIA MARATÓN CONCELLO DE VILAGARCIA DE AR...,2025-11-30,2,1,1,1,1
149,II MEDIA MARATÓN CONCELLO DE VILAGARCIA DE ARO...,2024-11-24,2,1,1,1,1
339,VI CARREIRA E CAMIÑADA CONTRA A VIOLENCIA DE X...,2025-11-30,2,1,1,1,1
375,VII TRAIL DAS BESTAS. CTO. XUNTA DE GALICIA TR...,2024-07-07,2,1,1,1,1


In [11]:
# Filas completas de esos eventos sospechosos, para decidir cómo fusionar
# (¿sumar ok_d/ok_h? ¿quedarnos con una fila?)
ejemplos = curses_limpio.merge(
    sospechosos[["nombre_carrera", "fecha"]], on=["nombre_carrera", "fecha"]
).sort_values(["nombre_carrera", "fecha"])

ejemplos.head(40)

,id,nombre_carrera,fecha,modalidad,finisher_d,finisher_h,municipio,distancia,tipo_modalidad,publico
27,K4joyLmY,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,CADETE MASC,0,43,Ribadavia,0.000,road running,Infantil
28,K4joyLmY,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,CADETE FEM,31,0,Ribadavia,0.000,road running,Infantil
29,K4joyLmY,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,INFANTIL MASC,3,56,Ribadavia,0.000,road running,Infantil
30,K4joyLmY,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,INFANTIL FEM,64,0,Ribadavia,0.000,road running,Infantil
31,K4joyLmY,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,ALEVÍN MASC,0,65,Ribadavia,0.000,road running,Infantil
32,K4joyLmY,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,ALEVÍN FEM,67,0,Ribadavia,0.000,road running,Infantil
33,K4joyLmY,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,BENXAMÍN MASC,0,70,Ribadavia,0.000,road running,Infantil
34,K4joyLmY,CAMPIONATO PROVINCIAL DE CROS ESCOLAR - OURENSE,2023-12-16,BENXAMÍN FEM,59,1,Ribadavia,0.000,road running,Infantil
0,bl3Ly8mP,CARREIRA E ANDAINA SOLIDARIA GATOCAN OROSO,2026-02-28,CARREIRA 5K,0,0,Oroso,5.000,road running,Absoluta/General
1,bl3Ly8mP,CARREIRA E ANDAINA SOLIDARIA GATOCAN OROSO,2026-02-28,CAMIÑADA,0,0,Oroso,5.000,road running,Absoluta/General


### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, xipgroc, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia` (antes `ok_d`/`ok_h` aquí). `fuente` es una constante ("carreirasgalegas") para identificar el origen al concatenar las 10 tablas. `municipio` se recupera de `lloc` (antes se descartaba); `comarca`/`provincia` no vienen en la fuente, así que las geocodificamos a partir de `municipio` (igual que hicimos en xipgroc/ccnorte/cronofinisher/sportmaniacs/cursescat/iter5/cruzandolameta). Lo que es propio solo de carreirasgalegas (`id`, `modalidad`) va al final.

In [12]:
# Geocodificamos "comarca"/"provincia" a partir de "municipio" (aquí ya es
# un nombre de municipio limpio, no un nombre de carrera con ruido, así
# que no hace falta ninguna heurística de extracción — geocodificamos
# directamente). Solo 146 municipios únicos, así que es rápido. Checkpoint
# propio en carreirasgalegas_ubicaciones.csv. En Galicia, "county" (comarca)
# de OpenStreetMap sí suele estar bien cubierto (a diferencia de otras
# zonas de España); "province" en cambio a menudo sale vacío para
# municipios pequeños y cae al fallback de comunidad autónoma ("Galicia").
import csv
import time


def geocodificar_ubicacion_municipios(municipios, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_ubic = out_path / "carreirasgalegas_ubicaciones.csv"

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["municipio"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} municipios ya geocodificados")

    geolocator = Nominatim(user_agent="carreirasgalegas_ubicaciones_claudia")

    municipios_unicos = list(dict.fromkeys(m for m in municipios if isinstance(m, str)))
    pendientes = [m for m in municipios_unicos if m not in cache]
    print(f"Municipios a geocodificar: {len(pendientes)} (de {len(municipios_unicos)} únicos)")

    campos = ["municipio", "comarca", "provincia"]
    write_header = not csv_ubic.exists()
    with open(csv_ubic, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        if write_header:
            writer.writeheader()

        for i, municipio in enumerate(pendientes, 1):
            fila = {"municipio": municipio, "comarca": None, "provincia": None}
            try:
                loc = geolocator.geocode(
                    f"{municipio}, Galicia, España", exactly_one=True, country_codes="es",
                    addressdetails=True, timeout=10,
                )
                if loc:
                    addr = loc.raw.get("address", {})
                    fila["comarca"] = addr.get("county")
                    fila["provincia"] = addr.get("province") or addr.get("state")
            except GeopyError as e:
                print(f"  [{municipio}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{municipio}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[municipio] = fila

            if i % 50 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de ubicaciones: {csv_ubic.resolve()}")
    return cache


OUT_DIR = Path("../../data/raw/carreirasgalegas")
_ubicaciones = geocodificar_ubicacion_municipios(curses_limpio["municipio"], out_dir=OUT_DIR)
curses_limpio["comarca"] = curses_limpio["municipio"].map(lambda m: _ubicaciones.get(m, {}).get("comarca"))
curses_limpio["provincia"] = curses_limpio["municipio"].map(lambda m: _ubicaciones.get(m, {}).get("provincia"))

print("Filas con provincia:", curses_limpio["provincia"].notna().sum(), "de", len(curses_limpio))
print("Filas con comarca:", curses_limpio["comarca"].notna().sum(), "de", len(curses_limpio))
curses_limpio[["municipio", "comarca", "provincia"]].drop_duplicates().sample(15, random_state=0)

Checkpoint: 146 municipios ya geocodificados
Municipios a geocodificar: 0 (de 146 únicos)
CSV de ubicaciones: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\carreirasgalegas_data_OK\carreirasgalegas_ubicaciones.csv
Filas con provincia: 2722 de 3192
Filas con comarca: 2640 de 3192


,municipio,comarca,provincia
44,Ordes - Oroso,Ordes,Galicia
154,Cuntis,Caldas,Galicia
186,Moaña,Morrazo,Galicia
2451,Baralla - O Corgo,NaN,NaN
3133,Ortigueira - Cariño,Ortegal,Galicia
341,Pontevedra,Pontevedra,Galicia
301,Castrelo de Miño,O Ribeiro,Ourense
150,Portas,Caldas,Galicia
2186,Campionato,NaN,NaN
165,A Baña,A Barcala,Galicia


In [13]:
# El geocodificado de arriba deja "comarca" bien resuelto, pero "provincia"
# se queda muchas veces vacío para municipios pequeños y cae al fallback
# de comunidad autónoma ("Galicia") — es justo lo que se veía al cruzar
# con la población del INE en el notebook de análisis (solo 13% de
# cobertura en esta fuente). Como toda la fuente es de Galicia, hay una
# forma más fiable que depender de OSM: cruzar el propio nombre de
# municipio contra el nomenclátor del INE, que ya viene con la provincia
# real y solo tiene 4 posibles en Galicia.
import glob
import unicodedata


def _normalizar(texto):
    if pd.isna(texto):
        return None
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return texto.upper().strip()


PROVINCIAS_GALICIA = {15: "A Coruña", 27: "Lugo", 32: "Ourense", 36: "Pontevedra"}

INE_DIR = Path("../../data/processed/union")
INE_PATH = glob.glob(str(INE_DIR / "DF_INE_Poblaciones_*.xlsx"))[0]
ine = pd.read_excel(INE_PATH, sheet_name="Sexo")
ine["_codigo"] = ine["Unidad Poblacional"].str.slice(0, 6)

ine_galicia = ine[(ine["_codigo"] == "000000") & (ine["Provincia"].isin(PROVINCIAS_GALICIA))].copy()
ine_galicia["municipio_norm"] = ine_galicia["Unidad Poblacional"].str.slice(6).str.strip().apply(_normalizar)
ine_galicia["provincia_ine"] = ine_galicia["Provincia"].map(PROVINCIAS_GALICIA)

duplicados = ine_galicia["municipio_norm"].duplicated().sum()
print(f"Municipios gallegos con nombre repetido en más de una provincia: {duplicados} "
      "(si es 0, cruzar solo por nombre es inequívoco)")

MUNICIPIO_A_PROVINCIA = dict(zip(ine_galicia["municipio_norm"], ine_galicia["provincia_ine"]))

provincia_ine = curses_limpio["municipio"].apply(_normalizar).map(MUNICIPIO_A_PROVINCIA)
antes = curses_limpio["provincia"].notna().sum()
curses_limpio["provincia"] = provincia_ine.fillna(curses_limpio["provincia"])
despues = curses_limpio["provincia"].notna().sum()

print(f"Filas con provincia: {antes} -> {despues} de {len(curses_limpio)}")
print(curses_limpio["provincia"].value_counts(dropna=False))


Municipios gallegos con nombre repetido en más de una provincia: 0 (si es 0, cruzar solo por nombre es inequívoco)
Filas con provincia: 2722 -> 2999 de 3192
provincia
A Coruña      1339
Galicia        511
Ourense        496
Pontevedra     451
Lugo           202
NaN            193
Name: count, dtype: int64


In [14]:
# Confirmado con el análisis de arriba: esas filas "sospechosas" son la
# misma categoría repartida en varias sub-categorías (p.ej. SUB8/SUB10/
# SUB12... todas cayendo en el mismo publico "Infantil") — las fusionamos
# en una sola fila por evento+distancia+tipo_modalidad+publico, sumando
# finisher_d/finisher_h. Añadimos "fuente" (constante, para identificar el
# origen al concatenar con las otras 5 tablas) y "dia_semana" (derivado de
# "fecha"), y reordenamos las columnas para que el esquema común (fuente,
# nombre_carrera, fecha, dia_semana, distancia, tipo_modalidad, publico,
# finisher_d, finisher_h, municipio, comarca, provincia) quede igual en
# las 6 fuentes, dejando lo propio de carreirasgalegas (id, modalidad) al
# final.
antes = len(curses_limpio)
curses_limpio = curses_limpio.groupby(
    ["nombre_carrera", "fecha", "distancia", "tipo_modalidad", "publico"], as_index=False
).agg(
    id=("id", "first"),
    municipio=("municipio", "first"),
    comarca=("comarca", "first"),
    provincia=("provincia", "first"),
    modalidad=("modalidad", lambda s: "; ".join(sorted(set(s.dropna())))),
    finisher_d=("finisher_d", "sum"),
    finisher_h=("finisher_h", "sum"),
)

curses_limpio["fuente"] = "carreirasgalegas"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia", "id", "modalidad"]
]

print(f"{antes - len(curses_limpio)} filas fusionadas ({antes} -> {len(curses_limpio)})")
curses_limpio.head()

395 filas fusionadas (3192 -> 2797)


,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,id,modalidad
0,carreirasgalegas,A CORUÑA EN MARCHA CONTRA O CANCRO,2022-05-29,Domingo,0.180,marcha,Infantil,0,0,A Coruña,A Coruña,Galicia,AOjGbZ95,ESCOLARES 5 (2015-Ata 3 anos)
1,carreirasgalegas,A CORUÑA EN MARCHA CONTRA O CANCRO,2022-05-29,Domingo,0.450,marcha,Infantil,0,0,A Coruña,A Coruña,Galicia,AOjGbZ95,ESCOLARES 4 (2013-2014)
2,carreirasgalegas,A CORUÑA EN MARCHA CONTRA O CANCRO,2022-05-29,Domingo,1.300,marcha,Absoluta/General,0,0,A Coruña,A Coruña,Galicia,AOjGbZ95,ESCOALRES 3 (2011-2012)
3,carreirasgalegas,A CORUÑA EN MARCHA CONTRA O CANCRO,2022-05-29,Domingo,1.799,marcha,Infantil,0,0,A Coruña,A Coruña,Galicia,AOjGbZ95,ESCOLARES 2 (2008-2009-2010)
4,carreirasgalegas,A CORUÑA EN MARCHA CONTRA O CANCRO,2022-05-29,Domingo,1.800,marcha,Infantil,0,0,A Coruña,A Coruña,Galicia,AOjGbZ95,ESCOLARES 1 (2006-2007-2005)


In [15]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/carreirasgalegas/DF_CARREIRASGALEGAS_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\carreirasgalegas_data_OK\DF_CARREIRASGALEGAS_LIMPIO.csv
